# DuckDB Database Build

## Objective

This notebook implements the relational database model defined in
`data/database_schema.dbml` and transforms the detailed API-Football fixture
JSON files into a reproducible DuckDB analytical database.

The notebook:

- validates and loads the raw detailed fixture files;
- transforms nested JSON blocks according to the grains established in
  `01_data_structure_inspection.ipynb`;
- standardizes timestamps, numeric values, and percentages;
- validates table grains and logical references before insertion;
- creates the physical DuckDB schema represented by the DBML file;
- loads all tables inside a transaction;
- creates the analytical views `fixture_teams` and
  `team_match_statistics_wide`;
- validates row counts and relational integrity after loading.

### Modeling principles

- Raw JSON files remain unchanged and are the source-of-truth layer.
- Players and coaches do **not** have a permanent `team_id` attribute.
  Team membership is contextual to each fixture.
- Competition identity is separated from competition-season.
- Team match statistics are physically stored in long format.
- Player match statistics remain wide because the observation grain is one
  player's performance in one fixture.

### Source-quality handling

Two source-data exceptions identified during implementation are handled
explicitly:

- a coach may have a valid `coach_id` while `coach_name` is unavailable;
  the identity is preserved and the descriptive name remains null;
- a lineup player may occasionally have no `player_id`. These entries are
  first matched against known identities using the conservative key
  `team_id + normalized player_name`. An ID is recovered only when that key
  maps to exactly one known player ID in the collected data. Ambiguous or
  unresolved entries are excluded from the relational core and retained in
  a transformation-quality report.

No synthetic player or coach identifiers are created.


## 1. Setup

In [1]:
import json
from pathlib import Path
from typing import Any

import duckdb
import pandas as pd

In [2]:
CWD = Path.cwd()

if (CWD / "pyproject.toml").exists():
    PROJECT_ROOT = CWD
elif (CWD.parent / "pyproject.toml").exists():
    PROJECT_ROOT = CWD.parent
else:
    raise FileNotFoundError("Could not locate the project root.")

RAW_DETAILS_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "api_football"
    / "fixtures"
    / "details"
)

DBML_PATH = PROJECT_ROOT / "data" / "database_schema.dbml"
DATABASE_DIR = PROJECT_ROOT / "database"
DATABASE_PATH = DATABASE_DIR / "football.duckdb"

DATABASE_DIR.mkdir(parents=True, exist_ok=True)

if not DBML_PATH.exists():
    raise FileNotFoundError(f"DBML schema not found: {DBML_PATH}")

if not RAW_DETAILS_DIR.exists():
    raise FileNotFoundError(
        f"Detailed fixtures directory not found: {RAW_DETAILS_DIR}"
    )

fixture_files = sorted(RAW_DETAILS_DIR.glob("fixture_*.json"))

if not fixture_files:
    raise FileNotFoundError(
        f"No fixture_*.json files found in {RAW_DETAILS_DIR}"
    )

print(f"Detailed fixture files found: {len(fixture_files)}")

Detailed fixture files found: 1222


## 2. Helper functions

In [3]:
def load_fixture_file(path: Path) -> dict[str, Any]:
    """Load and validate one detailed fixture response file."""
    with path.open("r", encoding="utf-8") as file:
        response = json.load(file)

    if not isinstance(response, list):
        raise TypeError(
            f"{path.name}: expected a list response, "
            f"found {type(response).__name__}."
        )

    if len(response) != 1:
        raise ValueError(
            f"{path.name}: expected exactly one fixture, found {len(response)}."
        )

    fixture = response[0]

    if not isinstance(fixture, dict):
        raise TypeError(f"{path.name}: fixture object is not a dict.")

    return fixture


def require_value(value: Any, label: str) -> Any:
    """Fail explicitly if a value required by the DB schema is missing."""
    if value is None:
        raise ValueError(f"Required source value is missing: {label}")
    return value


def to_float(value: Any) -> float | None:
    """Convert API numeric and percentage values to float."""
    if value is None:
        return None

    if isinstance(value, bool):
        return float(value)

    if isinstance(value, (int, float)):
        return float(value)

    if isinstance(value, str):
        cleaned = value.strip()
        if cleaned.lower() in {"", "none", "null", "nan"}:
            return None
        if cleaned.endswith("%"):
            cleaned = cleaned[:-1].strip()
        try:
            return float(cleaned)
        except ValueError as exc:
            raise ValueError(
                f"Could not parse numeric API value: {value!r}"
            ) from exc

    raise TypeError(
        f"Unsupported numeric value type: {type(value).__name__}"
    )


def to_int(value: Any) -> int | None:
    """Convert an integer-like API value to int."""
    numeric = to_float(value)
    if numeric is None:
        return None
    if not numeric.is_integer():
        raise ValueError(f"Expected integer-like value, found {value!r}.")
    return int(numeric)


def to_utc_naive_timestamp(value: Any):
    """Normalize source datetime to UTC for storage in DuckDB TIMESTAMP."""
    if value is None:
        return None
    parsed = pd.to_datetime(value, utc=True, errors="raise")
    return parsed.tz_convert(None).to_pydatetime()


def build_dataframe(records, columns):
    """Return a stable DataFrame schema even if there are no records."""
    return pd.DataFrame(records, columns=columns)


def resolve_dimension(df: pd.DataFrame, key: str) -> pd.DataFrame:
    if df.empty:
        return df.copy()

    attributes = [column for column in df.columns if column != key]

    for attribute in attributes:
        conflicts = (
            df.dropna(subset=[attribute])
            .groupby(key)[attribute]
            .nunique(dropna=True)
        )
        conflicting_ids = conflicts[conflicts > 1].index.tolist()
        if conflicting_ids:
            print(
                f"Warning: {attribute!r} has conflicting values for "
                f"{len(conflicting_ids)} {key}(s). "
                f"Examples: {conflicting_ids[:10]}"
            )

    def last_non_null(series):
        values = series.dropna()
        return None if values.empty else values.iloc[-1]

    return (
        df.groupby(key, as_index=False, sort=True)
        .agg({attribute: last_non_null for attribute in attributes})
    )

## 3. Load and revalidate the raw fixture structure

In [4]:
def normalize_api_id(value):
    """
    Normalize API entity identifiers.

    API-Football may use 0 when a real entity ID is unavailable.
    Zero (and negative values) are therefore treated as missing.
    """
    if value is None:
        return None

    try:
        normalized = int(value)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"Invalid API identifier: {value!r}"
        ) from exc

    if normalized <= 0:
        return None

    return normalized

In [5]:
raw_fixtures = [load_fixture_file(path) for path in fixture_files]

expected_top_level_keys = {
    "fixture",
    "league",
    "teams",
    "goals",
    "score",
    "events",
    "lineups",
    "statistics",
    "players",
}

structure_errors = []

for index, match in enumerate(raw_fixtures):
    actual_keys = set(match)
    missing_keys = expected_top_level_keys - actual_keys
    if missing_keys:
        structure_errors.append(
            {"index": index, "missing_keys": sorted(missing_keys)}
        )

if structure_errors:
    raise ValueError(
        "Fixture structure no longer matches notebook 01. "
        f"Examples: {structure_errors[:5]}"
    )

print(f"Fixtures loaded: {len(raw_fixtures)}")
print("Top-level structure validated.")

Fixtures loaded: 1222
Top-level structure validated.


## 4. Build dimension tables

In [6]:
def build_competitions(fixtures):
    columns = [
        "competition_id",
        "competition_name",
        "country",
        "logo_url",
        "flag_url",
    ]
    records = []

    for match in fixtures:
        league = match.get("league") or {}
        records.append(
            {
                "competition_id": require_value(
                    league.get("id"),
                    "league.id",
                ),
                "competition_name": require_value(
                    league.get("name"),
                    "league.name",
                ),
                "country": league.get("country"),
                "logo_url": league.get("logo"),
                "flag_url": league.get("flag"),
            }
        )

    return resolve_dimension(
        build_dataframe(records, columns),
        "competition_id",
    )


def build_competition_seasons(fixtures):
    columns = [
        "competition_id",
        "season",
    ]
    records = []

    for match in fixtures:
        league = match.get("league") or {}
        records.append(
            {
                "competition_id": require_value(
                    league.get("id"),
                    "league.id",
                ),
                "season": require_value(
                    league.get("season"),
                    "league.season",
                ),
            }
        )

    return (
        build_dataframe(records, columns)
        .drop_duplicates()
        .sort_values(
            ["competition_id", "season"]
        )
        .reset_index(drop=True)
    )


def build_teams(fixtures):
    columns = [
        "team_id",
        "team_name",
        "logo_url",
    ]
    records = []

    def append_team(team):
        if not team:
            return

        team_id = normalize_api_id(
            team.get("id")
        )

        if team_id is None:
            return

        records.append(
            {
                "team_id": team_id,
                "team_name": team.get("name"),
                "logo_url": team.get("logo"),
            }
        )

    for match in fixtures:
        teams = match.get("teams") or {}

        append_team(teams.get("home"))
        append_team(teams.get("away"))

        for lineup in match.get("lineups") or []:
            append_team(lineup.get("team"))

        for block in match.get("statistics") or []:
            append_team(block.get("team"))

        for block in match.get("players") or []:
            append_team(block.get("team"))

        for event in match.get("events") or []:
            append_team(event.get("team"))

    result = resolve_dimension(
        build_dataframe(records, columns),
        "team_id",
    )

    if result["team_name"].isna().any():
        raise ValueError(
            "At least one team_id has no team_name."
        )

    return result


def build_players(fixtures):
    columns = [
        "player_id",
        "player_name",
    ]
    records = []

    def append_player(player):
        if not player:
            return

        player_id = normalize_api_id(
            player.get("id")
        )

        if player_id is None:
            return

        records.append(
            {
                "player_id": player_id,
                "player_name": player.get("name"),
            }
        )

    for match in fixtures:
        for event in match.get("events") or []:
            append_player(event.get("player"))
            append_player(event.get("assist"))

        for lineup in match.get("lineups") or []:
            for item in lineup.get("startXI") or []:
                append_player(
                    item.get("player")
                )

            for item in lineup.get("substitutes") or []:
                append_player(
                    item.get("player")
                )

        for block in match.get("players") or []:
            for item in block.get("players") or []:
                append_player(
                    item.get("player")
                )

    result = resolve_dimension(
        build_dataframe(records, columns),
        "player_id",
    )

    if result["player_name"].isna().any():
        missing_ids = result.loc[
            result["player_name"].isna(),
            "player_id",
        ].tolist()

        raise ValueError(
            "Player IDs without a player name were found. "
            f"Examples: {missing_ids[:10]}"
        )

    return result


def build_coaches(fixtures):
    columns = [
        "coach_id",
        "coach_name",
    ]
    records = []

    for match in fixtures:
        for lineup in match.get("lineups") or []:
            coach = lineup.get("coach") or {}

            coach_id = normalize_api_id(
                coach.get("id")
            )

            if coach_id is None:
                continue

            records.append(
                {
                    "coach_id": coach_id,
                    "coach_name": coach.get("name"),
                }
            )

    result = resolve_dimension(
        build_dataframe(records, columns),
        "coach_id",
    )

    missing_names = (
        int(result["coach_name"].isna().sum())
        if not result.empty
        else 0
    )

    if missing_names:
        print(
            f"Warning: {missing_names} coach(es) have a valid "
            "coach_id but no coach_name available in the source data."
        )

    return result


def build_venues(fixtures):
    columns = [
        "venue_id",
        "venue_name",
        "city",
    ]
    records = []

    for match in fixtures:
        venue = (
            (match.get("fixture") or {})
            .get("venue")
            or {}
        )

        venue_id = normalize_api_id(
            venue.get("id")
        )

        if venue_id is None:
            continue

        records.append(
            {
                "venue_id": venue_id,
                "venue_name": venue.get("name"),
                "city": venue.get("city"),
            }
        )

    return resolve_dimension(
        build_dataframe(records, columns),
        "venue_id",
    )


competitions_df = build_competitions(
    raw_fixtures
)

competition_seasons_df = (
    build_competition_seasons(
        raw_fixtures
    )
)

teams_df = build_teams(
    raw_fixtures
)

players_df = build_players(
    raw_fixtures
)

coaches_df = build_coaches(
    raw_fixtures
)

venues_df = build_venues(
    raw_fixtures
)

dimension_summary_df = pd.DataFrame(
    {
        "table": [
            "competitions",
            "competition_seasons",
            "teams",
            "players",
            "coaches",
            "venues",
        ],
        "rows": [
            len(competitions_df),
            len(competition_seasons_df),
            len(teams_df),
            len(players_df),
            len(coaches_df),
            len(venues_df),
        ],
    }
)

dimension_summary_df

,table,rows
0,competitions,8
1,competition_seasons,22
2,teams,131
3,players,3650
4,coaches,179
5,venues,101


## 5. Build the fixture table

In [7]:
def build_fixtures(fixtures):
    columns = [
        "fixture_id",
        "competition_id",
        "season",
        "round",
        "match_date",
        "match_timestamp",
        "timezone",
        "status_short",
        "status_long",
        "elapsed",
        "referee",
        "venue_id",
        "home_team_id",
        "away_team_id",
        "home_goals",
        "away_goals",
        "halftime_home_goals",
        "halftime_away_goals",
        "fulltime_home_goals",
        "fulltime_away_goals",
        "extratime_home_goals",
        "extratime_away_goals",
        "penalty_home_goals",
        "penalty_away_goals",
    ]
    records = []

    for match in fixtures:
        fixture_info = match.get("fixture") or {}
        league = match.get("league") or {}
        teams = match.get("teams") or {}
        goals = match.get("goals") or {}
        score = match.get("score") or {}

        home = teams.get("home") or {}
        away = teams.get("away") or {}
        status = fixture_info.get("status") or {}
        venue = fixture_info.get("venue") or {}
        halftime = score.get("halftime") or {}
        fulltime = score.get("fulltime") or {}
        extratime = score.get("extratime") or {}
        penalty = score.get("penalty") or {}

        fixture_id = require_value(fixture_info.get("id"), "fixture.id")

        records.append(
            {
                "fixture_id": fixture_id,
                "competition_id": require_value(
                    league.get("id"), f"fixture {fixture_id}: league.id"
                ),
                "season": require_value(
                    league.get("season"), f"fixture {fixture_id}: league.season"
                ),
                "round": league.get("round"),
                "match_date": to_utc_naive_timestamp(fixture_info.get("date")),
                "match_timestamp": to_int(fixture_info.get("timestamp")),
                "timezone": fixture_info.get("timezone"),
                "status_short": status.get("short"),
                "status_long": status.get("long"),
                "elapsed": to_int(status.get("elapsed")),
                "referee": fixture_info.get("referee"),
                "venue_id": venue.get("id"),
                "home_team_id": require_value(
                    home.get("id"), f"fixture {fixture_id}: home team id"
                ),
                "away_team_id": require_value(
                    away.get("id"), f"fixture {fixture_id}: away team id"
                ),
                "home_goals": to_int(goals.get("home")),
                "away_goals": to_int(goals.get("away")),
                "halftime_home_goals": to_int(halftime.get("home")),
                "halftime_away_goals": to_int(halftime.get("away")),
                "fulltime_home_goals": to_int(fulltime.get("home")),
                "fulltime_away_goals": to_int(fulltime.get("away")),
                "extratime_home_goals": to_int(extratime.get("home")),
                "extratime_away_goals": to_int(extratime.get("away")),
                "penalty_home_goals": to_int(penalty.get("home")),
                "penalty_away_goals": to_int(penalty.get("away")),
            }
        )

    return build_dataframe(records, columns)


fixtures_df = build_fixtures(raw_fixtures)
print(f"Fixture rows: {len(fixtures_df)}")
fixtures_df.head()

Fixture rows: 1222


,fixture_id,competition_id,season,round,match_date,match_timestamp,timezone,status_short,status_long,elapsed,...,home_goals,away_goals,halftime_home_goals,halftime_away_goals,fulltime_home_goals,fulltime_away_goals,extratime_home_goals,extratime_away_goals,penalty_home_goals,penalty_away_goals
0,1004052,96,2022,Semi-finals,2023-04-26 19:30:00,1682537400,UTC,FT,Match Finished,90.0,...,1.0,2.0,1.0,1.0,1.0,2.0,NaN,NaN,NaN,NaN
1,1004053,96,2022,Semi-finals,2023-05-04 19:30:00,1683228600,UTC,AET,Match Finished,120.0,...,3.0,2.0,1.0,1.0,1.0,2.0,2.0,0.0,NaN,NaN
2,1010826,3,2022,Round of 16,2023-03-09 17:45:00,1678383900,UTC,FT,Match Finished,90.0,...,2.0,2.0,1.0,1.0,2.0,2.0,NaN,NaN,NaN,NaN
3,1010827,3,2022,Round of 16,2023-03-16 20:00:00,1678996800,UTC,PEN,Match Finished,120.0,...,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,3.0,5.0
4,1016043,3,2022,Quarter-finals,2023-04-13 19:00:00,1681412400,UTC,FT,Match Finished,90.0,...,1.0,0.0,0.0,0.0,1.0,0.0,NaN,NaN,NaN,NaN


## 6. Build match-detail and performance tables

The following tables implement the grains validated in
`01_data_structure_inspection.ipynb`:

- `events`: one row per fixture + event sequence;
- `team_match_statistics`: one row per fixture + team + statistic type;
- `lineups`: one row per fixture + team;
- `lineup_players`: one row per fixture + team + identified player;
- `player_match_statistics`: one row per fixture + player.

### Lineup player identity recovery

Some source lineup entries contain a player name and contextual information but
no `player_id`.

Because `player_id` is part of the relational identity of `lineup_players`,
the database does **not** accept null or synthetic player IDs.

The transformation therefore:

1. builds a reference map from observations where the API does provide a
   player ID;
2. matches missing lineup IDs by `team_id + normalized player_name`;
3. recovers an ID only when exactly one candidate exists;
4. records unresolved or ambiguous entries separately as transformation
   issues.

This preserves relational integrity without silently discarding source-quality
problems.


In [8]:
def normalize_player_name(value):
    """Normalize names conservatively for deterministic identity matching."""
    if value is None:
        return None

    normalized = " ".join(
        str(value).strip().split()
    )

    if not normalized:
        return None

    return normalized.casefold()


def build_player_identity_map(fixtures):
    columns = [
        "team_id",
        "player_name",
        "player_name_key",
        "player_id",
        "identity_source",
    ]

    records = []

    def append_identity(
        team_id,
        player,
        source,
    ):
        if team_id is None or not player:
            return

        player_id = normalize_api_id(
            player.get("id")
        )
        player_name = player.get("name")

        if (
            player_id is None
            or player_name is None
        ):
            return

        player_name_key = (
            normalize_player_name(
                player_name
            )
        )

        if player_name_key is None:
            return

        records.append(
            {
                "team_id": team_id,
                "player_name": player_name,
                "player_name_key": player_name_key,
                "player_id": player_id,
                "identity_source": source,
            }
        )

    for match in fixtures:
        # Lineups with valid IDs.
        for lineup in match.get("lineups") or []:
            team = lineup.get("team") or {}
            team_id = team.get("id")

            for item in (
                (lineup.get("startXI") or [])
                + (lineup.get("substitutes") or [])
            ):
                append_identity(
                    team_id,
                    item.get("player") or {},
                    "lineup",
                )

        # Player match-statistics blocks.
        for team_block in match.get("players") or []:
            team = team_block.get("team") or {}
            team_id = team.get("id")

            for item in team_block.get("players") or []:
                append_identity(
                    team_id,
                    item.get("player") or {},
                    "player_statistics",
                )

        # Events can provide additional known identities.
        for event in match.get("events") or []:
            team = event.get("team") or {}
            team_id = team.get("id")

            append_identity(
                team_id,
                event.get("player") or {},
                "event_player",
            )

            append_identity(
                team_id,
                event.get("assist") or {},
                "event_assist",
            )

    return (
        build_dataframe(
            records,
            columns,
        )
        .drop_duplicates()
        .reset_index(drop=True)
    )


def build_identity_candidates(identity_map_df):
    columns = [
        "team_id",
        "player_name_key",
        "candidate_ids",
        "candidate_count",
    ]

    if identity_map_df.empty:
        return pd.DataFrame(
            columns=columns
        )

    return (
        identity_map_df
        .groupby(
            [
                "team_id",
                "player_name_key",
            ],
            as_index=False,
        )
        .agg(
            candidate_ids=(
                "player_id",
                lambda series: sorted(
                    set(series.tolist())
                ),
            ),
            candidate_count=(
                "player_id",
                "nunique",
            ),
        )
        [columns]
    )


def inspect_missing_lineup_player_ids(
    fixtures,
):
    columns = [
        "fixture_id",
        "team_id",
        "team_name",
        "lineup_type",
        "player_id",
        "player_name",
        "number",
        "position",
        "grid",
    ]

    records = []

    for match in fixtures:
        fixture_id = (
            (match.get("fixture") or {})
            .get("id")
        )

        for lineup in match.get("lineups") or []:
            team = lineup.get("team") or {}
            team_id = team.get("id")
            team_name = team.get("name")

            groups = [
                (
                    "starter",
                    lineup.get("startXI") or [],
                ),
                (
                    "substitute",
                    lineup.get("substitutes") or [],
                ),
            ]

            for lineup_type, items in groups:
                for item in items:
                    player = item.get("player") or {}

                    if normalize_api_id(
                        player.get("id")
                    ) is not None:
                        continue

                    records.append(
                        {
                            "fixture_id": fixture_id,
                            "team_id": team_id,
                            "team_name": team_name,
                            "lineup_type": lineup_type,
                            "player_id": None,
                            "player_name": player.get("name"),
                            "number": player.get("number"),
                            "position": player.get("pos"),
                            "grid": player.get("grid"),
                        }
                    )

    return build_dataframe(records, columns)


def build_events(fixtures):
    columns = [
        "fixture_id",
        "event_sequence",
        "team_id",
        "player_id",
        "assist_id",
        "time_elapsed",
        "time_extra",
        "event_type",
        "event_detail",
        "comments",
    ]
    records = []

    for match in fixtures:
        fixture_id = require_value(
            (match.get("fixture") or {}).get("id"),
            "fixture.id",
        )

        for sequence, event in enumerate(
            match.get("events") or [],
            start=1,
        ):
            time = event.get("time") or {}
            team = event.get("team") or {}
            player = event.get("player") or {}
            assist = event.get("assist") or {}

            records.append(
                {
                    "fixture_id": fixture_id,
                    "event_sequence": sequence,
                    "team_id": team.get("id"),
                    "player_id": player.get("id"),
                    "assist_id": assist.get("id"),
                    "time_elapsed": to_int(
                        time.get("elapsed")
                    ),
                    "time_extra": to_int(
                        time.get("extra")
                    ),
                    "event_type": event.get("type"),
                    "event_detail": event.get("detail"),
                    "comments": event.get("comments"),
                }
            )

    return build_dataframe(records, columns)


def build_team_match_statistics(fixtures):
    columns = [
        "fixture_id",
        "team_id",
        "statistic_type",
        "statistic_value",
    ]
    records = []

    for match in fixtures:
        fixture_id = require_value(
            (match.get("fixture") or {}).get("id"),
            "fixture.id",
        )

        for block in match.get("statistics") or []:
            team = block.get("team") or {}

            team_id = require_value(
                team.get("id"),
                f"fixture {fixture_id}: statistics.team.id",
            )

            for statistic in block.get("statistics") or []:
                statistic_type = require_value(
                    statistic.get("type"),
                    (
                        f"fixture {fixture_id}, team {team_id}: "
                        "statistic.type"
                    ),
                )

                records.append(
                    {
                        "fixture_id": fixture_id,
                        "team_id": team_id,
                        "statistic_type": statistic_type,
                        "statistic_value": to_float(
                            statistic.get("value")
                        ),
                    }
                )

    return build_dataframe(records, columns)


def build_lineups(fixtures):
    columns = [
        "fixture_id",
        "team_id",
        "coach_id",
        "formation",
    ]

    records = []

    for match in fixtures:
        fixture_id = require_value(
            normalize_api_id(
                (match.get("fixture") or {}).get("id")
            ),
            "fixture.id",
        )

        for lineup in match.get("lineups") or []:
            team = lineup.get("team") or {}
            coach = lineup.get("coach") or {}

            team_id = require_value(
                normalize_api_id(
                    team.get("id")
                ),
                (
                    f"fixture {fixture_id}: "
                    "lineup.team.id"
                ),
            )

            coach_id = normalize_api_id(
                coach.get("id")
            )

            records.append(
                {
                    "fixture_id": fixture_id,
                    "team_id": team_id,
                    "coach_id": coach_id,
                    "formation": lineup.get("formation"),
                }
            )

    return build_dataframe(records, columns)


def build_lineup_players(fixtures, identity_candidates_df):
    processing_columns = [
        "fixture_id",
        "team_id",
        "player_id",
        "player_number",
        "position",
        "grid",
        "starter",
        "player_id_source",
    ]

    issue_columns = [
        "fixture_id",
        "team_id",
        "player_name",
        "player_number",
        "position",
        "grid",
        "starter",
        "candidate_ids",
        "candidate_count",
        "issue",
    ]

    records = []
    issues = []

    unique_identity_map = {}

    for row in identity_candidates_df.itertuples(
        index=False
    ):
        if row.candidate_count == 1:
            unique_identity_map[
                (
                    row.team_id,
                    row.player_name_key,
                )
            ] = row.candidate_ids[0]

    candidate_lookup = {
        (
            row.team_id,
            row.player_name_key,
        ): (
            row.candidate_ids,
            int(row.candidate_count),
        )
        for row in identity_candidates_df.itertuples(
            index=False
        )
    }

    def append_players(
        fixture_id,
        team_id,
        items,
        starter,
    ):
        for item in items:
            player = item.get("player") or {}

            player_id = normalize_api_id(
                player.get("id")
            )
            player_name = player.get("name")
            player_name_key = (
                normalize_player_name(
                    player_name
                )
            )

            player_id_source = "api"

            if player_id is None:
                lookup_key = (
                    team_id,
                    player_name_key,
                )

                player_id = (
                    unique_identity_map.get(
                        lookup_key
                    )
                    if player_name_key is not None
                    else None
                )

                if player_id is not None:
                    player_id_source = (
                        "recovered_team_name"
                    )
                else:
                    candidate_ids = []
                    candidate_count = 0

                    if (
                        player_name_key is not None
                        and lookup_key
                        in candidate_lookup
                    ):
                        (
                            candidate_ids,
                            candidate_count,
                        ) = candidate_lookup[
                            lookup_key
                        ]

                    issue_type = (
                        "ambiguous_player_identity"
                        if candidate_count > 1
                        else "unresolved_missing_player_id"
                    )

                    issues.append(
                        {
                            "fixture_id": fixture_id,
                            "team_id": team_id,
                            "player_name": player_name,
                            "player_number": to_int(
                                player.get("number")
                            ),
                            "position": player.get("pos"),
                            "grid": player.get("grid"),
                            "starter": starter,
                            "candidate_ids": candidate_ids,
                            "candidate_count": candidate_count,
                            "issue": issue_type,
                        }
                    )

                    continue

            records.append(
                {
                    "fixture_id": fixture_id,
                    "team_id": team_id,
                    "player_id": player_id,
                    "player_number": to_int(
                        player.get("number")
                    ),
                    "position": player.get("pos"),
                    "grid": player.get("grid"),
                    "starter": starter,
                    "player_id_source": player_id_source,
                }
            )

    for match in fixtures:
        fixture_id = require_value(
            (match.get("fixture") or {}).get("id"),
            "fixture.id",
        )

        for lineup in match.get("lineups") or []:
            team = lineup.get("team") or {}

            team_id = require_value(
                team.get("id"),
                (
                    f"fixture {fixture_id}: "
                    "lineup.team.id"
                ),
            )

            append_players(
                fixture_id,
                team_id,
                lineup.get("startXI") or [],
                True,
            )

            append_players(
                fixture_id,
                team_id,
                lineup.get("substitutes") or [],
                False,
            )

    return (
        build_dataframe(
            records,
            processing_columns,
        ),
        build_dataframe(
            issues,
            issue_columns,
        ),
    )

In [9]:
def build_player_match_statistics(
    fixtures,
    identity_candidates_df,
):
    """
    Build valid player-match statistics rows and a separate issue table.

    API player_id values <= 0 are treated as missing identifiers.

    Recovery is attempted only when:
        team_id + normalized player_name
    maps to exactly one known real player ID.

    Returns
    -------
    player_match_statistics_processing_df
        Valid rows including an audit-only `player_id_source` column.

    player_match_statistics_issues_df
        Rows that could not be assigned a unique real player ID.
    """
    processing_columns = [
        "fixture_id",
        "player_id",
        "team_id",
        "minutes",
        "player_number",
        "position",
        "rating",
        "captain",
        "substitute",
        "offsides",
        "shots_total",
        "shots_on",
        "goals_total",
        "goals_conceded",
        "goals_assists",
        "goals_saves",
        "passes_total",
        "passes_key",
        "passes_accuracy",
        "tackles_total",
        "tackles_blocks",
        "tackles_interceptions",
        "duels_total",
        "duels_won",
        "dribbles_attempts",
        "dribbles_success",
        "dribbles_past",
        "fouls_drawn",
        "fouls_committed",
        "cards_yellow",
        "cards_red",
        "penalty_won",
        "penalty_committed",
        "penalty_scored",
        "penalty_missed",
        "penalty_saved",
        "player_id_source",
    ]

    issue_columns = [
        "fixture_id",
        "team_id",
        "player_name",
        "player_number",
        "position",
        "candidate_ids",
        "candidate_count",
        "issue",
    ]

    records = []
    issues = []

    unique_identity_map = {}
    candidate_lookup = {}

    for row in identity_candidates_df.itertuples(
        index=False
    ):
        key = (
            row.team_id,
            row.player_name_key,
        )

        candidate_lookup[key] = (
            row.candidate_ids,
            int(row.candidate_count),
        )

        if row.candidate_count == 1:
            unique_identity_map[key] = (
                row.candidate_ids[0]
            )

    for match in fixtures:
        fixture_id = require_value(
            normalize_api_id(
                (match.get("fixture") or {}).get("id")
            ),
            "fixture.id",
        )

        for team_block in match.get("players") or []:
            team = team_block.get("team") or {}

            team_id = require_value(
                normalize_api_id(
                    team.get("id")
                ),
                (
                    f"fixture {fixture_id}: "
                    "players.team.id"
                ),
            )

            for item in team_block.get("players") or []:
                player = item.get("player") or {}

                player_name = player.get("name")
                player_name_key = (
                    normalize_player_name(
                        player_name
                    )
                )

                player_id = normalize_api_id(
                    player.get("id")
                )

                player_id_source = "api"

                if player_id is None:
                    lookup_key = (
                        team_id,
                        player_name_key,
                    )

                    player_id = (
                        unique_identity_map.get(
                            lookup_key
                        )
                        if player_name_key is not None
                        else None
                    )

                    if player_id is not None:
                        player_id_source = (
                            "recovered_team_name"
                        )
                    else:
                        candidate_ids = []
                        candidate_count = 0

                        if (
                            player_name_key is not None
                            and lookup_key in candidate_lookup
                        ):
                            (
                                candidate_ids,
                                candidate_count,
                            ) = candidate_lookup[
                                lookup_key
                            ]

                        issue = (
                            "ambiguous_player_identity"
                            if candidate_count > 1
                            else "unresolved_missing_player_id"
                        )

                        statistics_list = (
                            item.get("statistics") or []
                        )

                        stats = (
                            statistics_list[0]
                            if statistics_list
                            else {}
                        )

                        games = (
                            stats.get("games") or {}
                        )

                        issues.append(
                            {
                                "fixture_id": fixture_id,
                                "team_id": team_id,
                                "player_name": player_name,
                                "player_number": to_int(
                                    games.get("number")
                                ),
                                "position": games.get(
                                    "position"
                                ),
                                "candidate_ids": candidate_ids,
                                "candidate_count": (
                                    candidate_count
                                ),
                                "issue": issue,
                            }
                        )

                        continue

                statistics_list = (
                    item.get("statistics") or []
                )

                if len(statistics_list) > 1:
                    raise ValueError(
                        f"Fixture {fixture_id}, "
                        f"player {player_id}: "
                        "more than one statistics object "
                        "was found. Expected grain is "
                        "fixture + player."
                    )

                stats = (
                    statistics_list[0]
                    if statistics_list
                    else {}
                )

                games = stats.get("games") or {}
                shots = stats.get("shots") or {}
                goals = stats.get("goals") or {}
                passes = stats.get("passes") or {}
                tackles = stats.get("tackles") or {}
                duels = stats.get("duels") or {}
                dribbles = stats.get("dribbles") or {}
                fouls = stats.get("fouls") or {}
                cards = stats.get("cards") or {}
                penalty = stats.get("penalty") or {}

                penalty_committed = (
                    penalty.get("committed")
                )

                if penalty_committed is None:
                    penalty_committed = (
                        penalty.get("commited")
                    )

                records.append(
                    {
                        "fixture_id": fixture_id,
                        "player_id": player_id,
                        "team_id": team_id,
                        "minutes": to_int(
                            games.get("minutes")
                        ),
                        "player_number": to_int(
                            games.get("number")
                        ),
                        "position": games.get(
                            "position"
                        ),
                        "rating": to_float(
                            games.get("rating")
                        ),
                        "captain": games.get("captain"),
                        "substitute": games.get(
                            "substitute"
                        ),
                        "offsides": to_int(
                            stats.get("offsides")
                        ),
                        "shots_total": to_int(
                            shots.get("total")
                        ),
                        "shots_on": to_int(
                            shots.get("on")
                        ),
                        "goals_total": to_int(
                            goals.get("total")
                        ),
                        "goals_conceded": to_int(
                            goals.get("conceded")
                        ),
                        "goals_assists": to_int(
                            goals.get("assists")
                        ),
                        "goals_saves": to_int(
                            goals.get("saves")
                        ),
                        "passes_total": to_int(
                            passes.get("total")
                        ),
                        "passes_key": to_int(
                            passes.get("key")
                        ),
                        "passes_accuracy": to_float(
                            passes.get("accuracy")
                        ),
                        "tackles_total": to_int(
                            tackles.get("total")
                        ),
                        "tackles_blocks": to_int(
                            tackles.get("blocks")
                        ),
                        "tackles_interceptions": to_int(
                            tackles.get(
                                "interceptions"
                            )
                        ),
                        "duels_total": to_int(
                            duels.get("total")
                        ),
                        "duels_won": to_int(
                            duels.get("won")
                        ),
                        "dribbles_attempts": to_int(
                            dribbles.get("attempts")
                        ),
                        "dribbles_success": to_int(
                            dribbles.get("success")
                        ),
                        "dribbles_past": to_int(
                            dribbles.get("past")
                        ),
                        "fouls_drawn": to_int(
                            fouls.get("drawn")
                        ),
                        "fouls_committed": to_int(
                            fouls.get("committed")
                        ),
                        "cards_yellow": to_int(
                            cards.get("yellow")
                        ),
                        "cards_red": to_int(
                            cards.get("red")
                        ),
                        "penalty_won": to_int(
                            penalty.get("won")
                        ),
                        "penalty_committed": to_int(
                            penalty_committed
                        ),
                        "penalty_scored": to_int(
                            penalty.get("scored")
                        ),
                        "penalty_missed": to_int(
                            penalty.get("missed")
                        ),
                        "penalty_saved": to_int(
                            penalty.get("saved")
                        ),
                        "player_id_source": (
                            player_id_source
                        ),
                    }
                )

    return (
        build_dataframe(
            records,
            processing_columns,
        ),
        build_dataframe(
            issues,
            issue_columns,
        ),
    )


# ---------------------------------------------------------------
# Identity reference and raw missing-ID inspection
# ---------------------------------------------------------------

player_identity_map_df = (
    build_player_identity_map(
        raw_fixtures
    )
)

player_identity_candidates_df = (
    build_identity_candidates(
        player_identity_map_df
    )
)

missing_lineup_player_ids_df = (
    inspect_missing_lineup_player_ids(
        raw_fixtures
    )
)

print(
    "Raw lineup entries without a valid player_id:",
    len(missing_lineup_player_ids_df),
)


# ---------------------------------------------------------------
# Build match-detail tables
# ---------------------------------------------------------------

events_df = build_events(
    raw_fixtures
)

team_match_statistics_df = (
    build_team_match_statistics(
        raw_fixtures
    )
)

lineups_df = build_lineups(
    raw_fixtures
)

(
    lineup_players_processing_df,
    lineup_player_issues_df,
) = build_lineup_players(
    raw_fixtures,
    player_identity_candidates_df,
)

lineup_players_df = (
    lineup_players_processing_df[
        [
            "fixture_id",
            "team_id",
            "player_id",
            "player_number",
            "position",
            "grid",
            "starter",
        ]
    ]
    .copy()
)

(
    player_match_statistics_processing_df,
    player_match_statistics_issues_df,
) = build_player_match_statistics(
    raw_fixtures,
    player_identity_candidates_df,
)

player_match_statistics_df = (
    player_match_statistics_processing_df
    .drop(
        columns="player_id_source"
    )
    .copy()
)


# ---------------------------------------------------------------
# Recovery and transformation-quality summary
# ---------------------------------------------------------------

lineup_recovery_summary_df = (
    lineup_players_processing_df[
        "player_id_source"
    ]
    .value_counts(dropna=False)
    .rename_axis("player_id_source")
    .reset_index(name="rows")
)

player_stats_recovery_summary_df = (
    player_match_statistics_processing_df[
        "player_id_source"
    ]
    .value_counts(dropna=False)
    .rename_axis("player_id_source")
    .reset_index(name="rows")
)

recovered_lineup_ids = int(
    (
        lineup_players_processing_df[
            "player_id_source"
        ]
        == "recovered_team_name"
    ).sum()
)

recovered_player_stat_ids = int(
    (
        player_match_statistics_processing_df[
            "player_id_source"
        ]
        == "recovered_team_name"
    ).sum()
)

coach_missing_names = (
    int(
        coaches_df["coach_name"]
        .isna()
        .sum()
    )
    if not coaches_df.empty
    else 0
)

transformation_quality_summary_df = pd.DataFrame(
    [
        {
            "quality_check": (
                "coach_id with missing coach_name"
            ),
            "rows": coach_missing_names,
        },
        {
            "quality_check": (
                "raw lineup entries without valid player_id"
            ),
            "rows": len(
                missing_lineup_player_ids_df
            ),
        },
        {
            "quality_check": (
                "lineup player IDs recovered uniquely"
            ),
            "rows": recovered_lineup_ids,
        },
        {
            "quality_check": (
                "lineup player IDs unresolved/ambiguous"
            ),
            "rows": len(
                lineup_player_issues_df
            ),
        },
        {
            "quality_check": (
                "player-stat IDs recovered uniquely"
            ),
            "rows": recovered_player_stat_ids,
        },
        {
            "quality_check": (
                "player-stat IDs unresolved/ambiguous"
            ),
            "rows": len(
                player_match_statistics_issues_df
            ),
        },
    ]
)

print("Lineup player-ID recovery:")
display(
    lineup_recovery_summary_df
)

print("Player-statistics player-ID recovery:")
display(
    player_stats_recovery_summary_df
)

print("Transformation quality:")
display(
    transformation_quality_summary_df
)

if not lineup_player_issues_df.empty:
    print(
        "Unresolved/ambiguous lineup-player entries:"
    )
    display(
        lineup_player_issues_df
    )

if not player_match_statistics_issues_df.empty:
    print(
        "Unresolved/ambiguous player-statistic entries:"
    )
    display(
        player_match_statistics_issues_df
    )


transformation_summary_df = pd.DataFrame(
    {
        "table": [
            "fixtures",
            "events",
            "team_match_statistics",
            "lineups",
            "lineup_players",
            "player_match_statistics",
        ],
        "rows": [
            len(fixtures_df),
            len(events_df),
            len(team_match_statistics_df),
            len(lineups_df),
            len(lineup_players_df),
            len(player_match_statistics_df),
        ],
    }
)

transformation_summary_df

Raw lineup entries without a valid player_id: 37
Lineup player-ID recovery:


,player_id_source,rows
0,api,47590
1,recovered_team_name,15


Player-statistics player-ID recovery:


,player_id_source,rows
0,api,44170
1,recovered_team_name,21


Transformation quality:


...

Unresolved/ambiguous lineup-player entries:


,fixture_id,team_id,player_name,player_number,position,grid,starter,candidate_ids,candidate_count,issue
0,1030529,551,Thierno Barry,11.0,F,4:1,True,[],0,unresolved_missing_player_id
1,1030529,551,Junior Ze,32.0,F,NaN,False,[],0,unresolved_missing_player_id
2,1036858,228,Leonardo Barroso,NaN,D,NaN,False,[],0,unresolved_missing_player_id
3,1036858,548,Hugo Martín,NaN,NaN,NaN,False,[],0,unresolved_missing_player_id
4,1036879,533,Víctor Moreno,29.0,M,NaN,False,[],0,unresolved_missing_player_id
5,1047250,43,Xavier Benjamin,96.0,D,NaN,False,[],0,unresolved_missing_player_id
6,1051827,43,Xavier Benjamin,99.0,D,2:4,True,[],0,unresolved_missing_player_id
7,1058343,217,Hadji,NaN,M,NaN,False,[],0,unresolved_missing_player_id
8,1072479,742,Tiago Ferreira,78.0,F,NaN,False,[],0,unresolved_missing_player_id
9,1072609,2939,Abdulaziz Al-Aliwa,46.0,M,NaN,False,[],0,unresolved_missing_player_id


Unresolved/ambiguous player-statistic entries:


,fixture_id,team_id,player_name,player_number,position,candidate_ids,candidate_count,issue
0,1136319,4763,Luis Machado,7,F,[],0,unresolved_missing_player_id
1,1136319,4763,Didier Mosquera,70,D,[],0,unresolved_missing_player_id
2,1219571,209,Jan Plug,45,D,[],0,unresolved_missing_player_id


,table,rows
0,fixtures,1222
1,events,21214
2,team_match_statistics,38432
3,lineups,2356
4,lineup_players,47605
5,player_match_statistics,44191


## 7. Validate table grains and logical references before loading

Only relationally valid rows enter the physical database.

Entries that could not be assigned a real, unique player ID remain available in
`lineup_player_issues_df` for audit and future enrichment, while the original
raw JSON remains unchanged.

The checks below validate the physical core that will be loaded into DuckDB.


In [10]:
# ---------------------------------------------------------------
# Core entity-ID validation
# ---------------------------------------------------------------

entity_id_checks = {
    "competitions": (
        competitions_df,
        "competition_id",
    ),
    "teams": (
        teams_df,
        "team_id",
    ),
    "players": (
        players_df,
        "player_id",
    ),
    "coaches": (
        coaches_df,
        "coach_id",
    ),
    "venues": (
        venues_df,
        "venue_id",
    ),
    "fixtures": (
        fixtures_df,
        "fixture_id",
    ),
}

for entity_name, (
    entity_df,
    id_column,
) in entity_id_checks.items():
    if entity_df.empty:
        continue

    invalid_ids = entity_df[
        entity_df[id_column] <= 0
    ]

    if not invalid_ids.empty:
        raise ValueError(
            f"{entity_name} contains invalid "
            f"{id_column} values <= 0."
        )

print("All core entity IDs are positive.")

TABLES = {
    "competitions": competitions_df,
    "competition_seasons": competition_seasons_df,
    "teams": teams_df,
    "players": players_df,
    "coaches": coaches_df,
    "venues": venues_df,
    "fixtures": fixtures_df,
    "events": events_df,
    "team_match_statistics": team_match_statistics_df,
    "lineups": lineups_df,
    "lineup_players": lineup_players_df,
    "player_match_statistics": player_match_statistics_df,
}

PRIMARY_KEYS = {
    "competitions": ["competition_id"],
    "competition_seasons": ["competition_id", "season"],
    "teams": ["team_id"],
    "players": ["player_id"],
    "coaches": ["coach_id"],
    "venues": ["venue_id"],
    "fixtures": ["fixture_id"],
    "events": ["fixture_id", "event_sequence"],
    "team_match_statistics": ["fixture_id", "team_id", "statistic_type"],
    "lineups": ["fixture_id", "team_id"],
    "lineup_players": ["fixture_id", "team_id", "player_id"],
    "player_match_statistics": ["fixture_id", "player_id"],
}

validation_rows = []

for table_name, keys in PRIMARY_KEYS.items():
    df = TABLES[table_name]
    null_keys = int(df[keys].isna().any(axis=1).sum()) if not df.empty else 0
    duplicates = int(df.duplicated(subset=keys, keep=False).sum()) if not df.empty else 0
    validation_rows.append(
        {
            "table": table_name,
            "rows": len(df),
            "null_key_rows": null_keys,
            "duplicate_key_rows": duplicates,
        }
    )

validation_df = pd.DataFrame(validation_rows)
validation_df

All core entity IDs are positive.


,table,rows,null_key_rows,duplicate_key_rows
0,competitions,8,0,0
1,competition_seasons,22,0,0
2,teams,131,0,0
3,players,3650,0,0
4,coaches,179,0,0
5,venues,101,0,0
6,fixtures,1222,0,0
7,events,21214,0,0
8,team_match_statistics,38432,0,0
9,lineups,2356,0,0


In [11]:
invalid = validation_df[
    (validation_df["null_key_rows"] > 0)
    | (validation_df["duplicate_key_rows"] > 0)
]

if not invalid.empty:
    raise ValueError(
        "One or more transformed tables violate the expected grain:\n"
        + invalid.to_string(index=False)
    )

if len(fixtures_df) != len(raw_fixtures):
    raise ValueError(
        "Fixture row count differs from the number of raw fixture files."
    )


def assert_subset(child_values, parent_values, relationship):
    child = set(child_values.dropna().tolist())
    parent = set(parent_values.dropna().tolist())
    missing = child - parent
    if missing:
        raise ValueError(
            f"Broken logical reference {relationship}. "
            f"Missing parent IDs: {sorted(missing)[:10]}"
        )


assert_subset(fixtures_df["home_team_id"], teams_df["team_id"], "home team")
assert_subset(fixtures_df["away_team_id"], teams_df["team_id"], "away team")
assert_subset(fixtures_df["venue_id"], venues_df["venue_id"], "venue")

for name in [
    "events",
    "team_match_statistics",
    "lineups",
    "lineup_players",
    "player_match_statistics",
]:
    assert_subset(TABLES[name]["fixture_id"], fixtures_df["fixture_id"], f"{name} -> fixtures")

for name in [
    "events",
    "team_match_statistics",
    "lineups",
    "lineup_players",
    "player_match_statistics",
]:
    assert_subset(TABLES[name]["team_id"], teams_df["team_id"], f"{name} -> teams")

assert_subset(events_df["player_id"], players_df["player_id"], "events.player_id")
assert_subset(events_df["assist_id"], players_df["player_id"], "events.assist_id")
assert_subset(lineup_players_df["player_id"], players_df["player_id"], "lineup players")
assert_subset(player_match_statistics_df["player_id"], players_df["player_id"], "player statistics")
assert_subset(lineups_df["coach_id"], coaches_df["coach_id"], "lineups.coach_id")

competition_seasons = set(
    map(tuple, competition_seasons_df[["competition_id", "season"]].to_numpy())
)
fixture_competition_seasons = set(
    map(tuple, fixtures_df[["competition_id", "season"]].to_numpy())
)

if fixture_competition_seasons - competition_seasons:
    raise ValueError("Fixtures contain unknown competition-season pairs.")

print("All table grains and logical references validated.")

All table grains and logical references validated.


## 8. Create the physical DuckDB schema

In [12]:
CREATE_TABLES_SQL = r"""
CREATE TABLE competitions (
    competition_id BIGINT PRIMARY KEY,
    competition_name VARCHAR NOT NULL,
    country VARCHAR,
    logo_url VARCHAR,
    flag_url VARCHAR
);

CREATE TABLE competition_seasons (
    competition_id BIGINT NOT NULL,
    season INTEGER NOT NULL,
    PRIMARY KEY (competition_id, season),
    FOREIGN KEY (competition_id) REFERENCES competitions(competition_id)
);

CREATE TABLE teams (
    team_id BIGINT PRIMARY KEY,
    team_name VARCHAR NOT NULL,
    logo_url VARCHAR
);

CREATE TABLE players (
    player_id BIGINT PRIMARY KEY,
    player_name VARCHAR NOT NULL
);

CREATE TABLE coaches (
    coach_id BIGINT PRIMARY KEY,
    coach_name VARCHAR
);

CREATE TABLE venues (
    venue_id BIGINT PRIMARY KEY,
    venue_name VARCHAR,
    city VARCHAR
);

CREATE TABLE fixtures (
    fixture_id BIGINT PRIMARY KEY,
    competition_id BIGINT NOT NULL,
    season INTEGER NOT NULL,
    round VARCHAR,
    match_date TIMESTAMP,
    match_timestamp BIGINT,
    timezone VARCHAR,
    status_short VARCHAR,
    status_long VARCHAR,
    elapsed INTEGER,
    referee VARCHAR,
    venue_id BIGINT,
    home_team_id BIGINT NOT NULL,
    away_team_id BIGINT NOT NULL,
    home_goals INTEGER,
    away_goals INTEGER,
    halftime_home_goals INTEGER,
    halftime_away_goals INTEGER,
    fulltime_home_goals INTEGER,
    fulltime_away_goals INTEGER,
    extratime_home_goals INTEGER,
    extratime_away_goals INTEGER,
    penalty_home_goals INTEGER,
    penalty_away_goals INTEGER,
    FOREIGN KEY (competition_id, season)
        REFERENCES competition_seasons(competition_id, season),
    FOREIGN KEY (venue_id) REFERENCES venues(venue_id),
    FOREIGN KEY (home_team_id) REFERENCES teams(team_id),
    FOREIGN KEY (away_team_id) REFERENCES teams(team_id)
);

CREATE TABLE events (
    fixture_id BIGINT NOT NULL,
    event_sequence INTEGER NOT NULL,
    team_id BIGINT,
    player_id BIGINT,
    assist_id BIGINT,
    time_elapsed INTEGER,
    time_extra INTEGER,
    event_type VARCHAR,
    event_detail VARCHAR,
    comments VARCHAR,
    PRIMARY KEY (fixture_id, event_sequence),
    FOREIGN KEY (fixture_id) REFERENCES fixtures(fixture_id),
    FOREIGN KEY (team_id) REFERENCES teams(team_id),
    FOREIGN KEY (player_id) REFERENCES players(player_id),
    FOREIGN KEY (assist_id) REFERENCES players(player_id)
);

CREATE TABLE team_match_statistics (
    fixture_id BIGINT NOT NULL,
    team_id BIGINT NOT NULL,
    statistic_type VARCHAR NOT NULL,
    statistic_value DOUBLE,
    PRIMARY KEY (fixture_id, team_id, statistic_type),
    FOREIGN KEY (fixture_id) REFERENCES fixtures(fixture_id),
    FOREIGN KEY (team_id) REFERENCES teams(team_id)
);

CREATE TABLE lineups (
    fixture_id BIGINT NOT NULL,
    team_id BIGINT NOT NULL,
    coach_id BIGINT,
    formation VARCHAR,
    PRIMARY KEY (fixture_id, team_id),
    FOREIGN KEY (fixture_id) REFERENCES fixtures(fixture_id),
    FOREIGN KEY (team_id) REFERENCES teams(team_id),
    FOREIGN KEY (coach_id) REFERENCES coaches(coach_id)
);

CREATE TABLE lineup_players (
    fixture_id BIGINT NOT NULL,
    team_id BIGINT NOT NULL,
    player_id BIGINT NOT NULL,
    player_number INTEGER,
    position VARCHAR,
    grid VARCHAR,
    starter BOOLEAN NOT NULL,
    PRIMARY KEY (fixture_id, team_id, player_id),
    FOREIGN KEY (fixture_id) REFERENCES fixtures(fixture_id),
    FOREIGN KEY (team_id) REFERENCES teams(team_id),
    FOREIGN KEY (player_id) REFERENCES players(player_id)
);

CREATE TABLE player_match_statistics (
    fixture_id BIGINT NOT NULL,
    player_id BIGINT NOT NULL,
    team_id BIGINT NOT NULL,
    minutes INTEGER,
    player_number INTEGER,
    position VARCHAR,
    rating DOUBLE,
    captain BOOLEAN,
    substitute BOOLEAN,
    offsides INTEGER,
    shots_total INTEGER,
    shots_on INTEGER,
    goals_total INTEGER,
    goals_conceded INTEGER,
    goals_assists INTEGER,
    goals_saves INTEGER,
    passes_total INTEGER,
    passes_key INTEGER,
    passes_accuracy DOUBLE,
    tackles_total INTEGER,
    tackles_blocks INTEGER,
    tackles_interceptions INTEGER,
    duels_total INTEGER,
    duels_won INTEGER,
    dribbles_attempts INTEGER,
    dribbles_success INTEGER,
    dribbles_past INTEGER,
    fouls_drawn INTEGER,
    fouls_committed INTEGER,
    cards_yellow INTEGER,
    cards_red INTEGER,
    penalty_won INTEGER,
    penalty_committed INTEGER,
    penalty_scored INTEGER,
    penalty_missed INTEGER,
    penalty_saved INTEGER,
    PRIMARY KEY (fixture_id, player_id),
    FOREIGN KEY (fixture_id) REFERENCES fixtures(fixture_id),
    FOREIGN KEY (team_id) REFERENCES teams(team_id),
    FOREIGN KEY (player_id) REFERENCES players(player_id)
);
"""

REBUILD_DATABASE = True

existing_connection = globals().get("con")

if existing_connection is not None:
    try:
        existing_connection.close()
    except Exception:
        pass

if DATABASE_PATH.exists():
    if not REBUILD_DATABASE:
        raise FileExistsError(
            f"Database already exists: {DATABASE_PATH}. "
            "Set REBUILD_DATABASE = True to rebuild it."
        )
    DATABASE_PATH.unlink()

wal_path = Path(f"{DATABASE_PATH}.wal")
if wal_path.exists():
    wal_path.unlink()

con = duckdb.connect(str(DATABASE_PATH))
con.execute(CREATE_TABLES_SQL)

con.sql("SHOW TABLES").df()

,name
0,coaches
1,competition_seasons
2,competitions
3,events
4,fixtures
5,lineup_players
6,lineups
7,player_match_statistics
8,players
9,team_match_statistics


## 9. Load transformed tables inside one transaction

In [13]:
LOAD_ORDER = [
    "competitions",
    "competition_seasons",
    "teams",
    "players",
    "coaches",
    "venues",
    "fixtures",
    "events",
    "team_match_statistics",
    "lineups",
    "lineup_players",
    "player_match_statistics",
]


def insert_dataframe(connection, table_name, df):
    staging_name = f"_staging_{table_name}"
    connection.register(staging_name, df)
    try:
        connection.execute(
            f'INSERT INTO "{table_name}" BY NAME '
            f'SELECT * FROM "{staging_name}"'
        )
    finally:
        connection.unregister(staging_name)


con.execute("BEGIN TRANSACTION")
try:
    for table_name in LOAD_ORDER:
        insert_dataframe(con, table_name, TABLES[table_name])
    con.execute("COMMIT")
except Exception:
    con.execute("ROLLBACK")
    raise

print("All physical tables loaded successfully.")

All physical tables loaded successfully.


## 10. Create analytical views

In [14]:
CREATE_VIEWS_SQL = r"""
CREATE OR REPLACE VIEW fixture_teams AS
SELECT
    fixture_id,
    competition_id,
    season,
    match_date,
    home_team_id AS team_id,
    away_team_id AS opponent_id,
    'home' AS side,
    home_goals AS goals_for,
    away_goals AS goals_against,
    CASE
        WHEN home_goals IS NULL OR away_goals IS NULL THEN NULL
        WHEN home_goals > away_goals THEN 'W'
        WHEN home_goals = away_goals THEN 'D'
        ELSE 'L'
    END AS result,
    CASE
        WHEN home_goals IS NULL OR away_goals IS NULL THEN NULL
        WHEN home_goals > away_goals THEN 3
        WHEN home_goals = away_goals THEN 1
        ELSE 0
    END AS points,
    CASE
        WHEN home_goals IS NULL OR away_goals IS NULL THEN NULL
        ELSE home_goals - away_goals
    END AS goal_difference
FROM fixtures

UNION ALL

SELECT
    fixture_id,
    competition_id,
    season,
    match_date,
    away_team_id AS team_id,
    home_team_id AS opponent_id,
    'away' AS side,
    away_goals AS goals_for,
    home_goals AS goals_against,
    CASE
        WHEN home_goals IS NULL OR away_goals IS NULL THEN NULL
        WHEN away_goals > home_goals THEN 'W'
        WHEN away_goals = home_goals THEN 'D'
        ELSE 'L'
    END AS result,
    CASE
        WHEN home_goals IS NULL OR away_goals IS NULL THEN NULL
        WHEN away_goals > home_goals THEN 3
        WHEN away_goals = home_goals THEN 1
        ELSE 0
    END AS points,
    CASE
        WHEN home_goals IS NULL OR away_goals IS NULL THEN NULL
        ELSE away_goals - home_goals
    END AS goal_difference
FROM fixtures;


CREATE OR REPLACE VIEW team_match_statistics_wide AS
SELECT
    fixture_id,
    team_id,
    MAX(CASE WHEN statistic_type = 'Shots on Goal' THEN statistic_value END) AS shots_on_goal,
    MAX(CASE WHEN statistic_type = 'Shots off Goal' THEN statistic_value END) AS shots_off_goal,
    MAX(CASE WHEN statistic_type = 'Total Shots' THEN statistic_value END) AS total_shots,
    MAX(CASE WHEN statistic_type = 'Blocked Shots' THEN statistic_value END) AS blocked_shots,
    MAX(CASE WHEN statistic_type = 'Shots insidebox' THEN statistic_value END) AS shots_inside_box,
    MAX(CASE WHEN statistic_type = 'Shots outsidebox' THEN statistic_value END) AS shots_outside_box,
    MAX(CASE WHEN statistic_type = 'Fouls' THEN statistic_value END) AS fouls,
    MAX(CASE WHEN statistic_type = 'Corner Kicks' THEN statistic_value END) AS corner_kicks,
    MAX(CASE WHEN statistic_type = 'Offsides' THEN statistic_value END) AS offsides,
    MAX(CASE WHEN statistic_type = 'Ball Possession' THEN statistic_value END) AS ball_possession,
    MAX(CASE WHEN statistic_type = 'Yellow Cards' THEN statistic_value END) AS yellow_cards,
    MAX(CASE WHEN statistic_type = 'Red Cards' THEN statistic_value END) AS red_cards,
    MAX(CASE WHEN statistic_type = 'Goalkeeper Saves' THEN statistic_value END) AS goalkeeper_saves,
    MAX(CASE WHEN statistic_type = 'Total passes' THEN statistic_value END) AS total_passes,
    MAX(CASE WHEN statistic_type = 'Passes accurate' THEN statistic_value END) AS passes_accurate,
    MAX(CASE WHEN statistic_type = 'Passes %' THEN statistic_value END) AS passes_pct,
    MAX(CASE WHEN statistic_type = 'expected_goals' THEN statistic_value END) AS expected_goals,
    MAX(CASE WHEN statistic_type = 'goals_prevented' THEN statistic_value END) AS goals_prevented
FROM team_match_statistics
GROUP BY fixture_id, team_id;
"""

con.execute(CREATE_VIEWS_SQL)
print("Analytical views created.")

Analytical views created.


## 11. Validate the resulting database

In [15]:
row_counts = []

for table_name in LOAD_ORDER:
    rows = con.execute(
        f'SELECT COUNT(*) FROM "{table_name}"'
    ).fetchone()[0]
    row_counts.append({"table": table_name, "rows": rows})

row_counts_df = pd.DataFrame(row_counts)
row_counts_df

,table,rows
0,competitions,8
1,competition_seasons,22
2,teams,131
3,players,3650
4,coaches,179
5,venues,101
6,fixtures,1222
7,events,21214
8,team_match_statistics,38432
9,lineups,2356


In [16]:
expected_counts = {name: len(TABLES[name]) for name in LOAD_ORDER}
actual_counts = dict(zip(row_counts_df["table"], row_counts_df["rows"]))

mismatches = {
    name: {"expected": expected_counts[name], "actual": actual_counts[name]}
    for name in LOAD_ORDER
    if expected_counts[name] != actual_counts[name]
}

if mismatches:
    raise ValueError(f"Database row-count mismatches: {mismatches}")

fixture_team_rows = con.execute(
    "SELECT COUNT(*) FROM fixture_teams"
).fetchone()[0]

if fixture_team_rows != 2 * len(fixtures_df):
    raise ValueError(
        "fixture_teams should contain exactly two rows per fixture."
    )

orphan_checks = {
    "events -> fixtures": """
        SELECT COUNT(*) FROM events e
        LEFT JOIN fixtures f ON e.fixture_id = f.fixture_id
        WHERE f.fixture_id IS NULL
    """,
    "lineup players -> players": """
        SELECT COUNT(*) FROM lineup_players lp
        LEFT JOIN players p ON lp.player_id = p.player_id
        WHERE p.player_id IS NULL
    """,
    "player statistics -> players": """
        SELECT COUNT(*) FROM player_match_statistics ps
        LEFT JOIN players p ON ps.player_id = p.player_id
        WHERE p.player_id IS NULL
    """,
}

orphan_results = []
for relationship, query in orphan_checks.items():
    count = con.execute(query).fetchone()[0]
    orphan_results.append(
        {"relationship": relationship, "orphan_rows": count}
    )

orphan_results_df = pd.DataFrame(orphan_results)

if (orphan_results_df["orphan_rows"] > 0).any():
    raise ValueError("Relational integrity validation found orphan rows.")

print("Database row counts and relational integrity validated.")
orphan_results_df

Database row counts and relational integrity validated.


,relationship,orphan_rows
0,events -> fixtures,0
1,lineup players -> players,0
2,player statistics -> players,0


In [17]:
observed_statistic_types = con.sql(
    """
    SELECT statistic_type, COUNT(*) AS observations
    FROM team_match_statistics
    GROUP BY statistic_type
    ORDER BY statistic_type
    """
).df()

print(f"Distinct team statistic types: {len(observed_statistic_types)}")
observed_statistic_types

Distinct team statistic types: 18


,statistic_type,observations
0,Ball Possession,2238
1,Blocked Shots,2238
2,Corner Kicks,2238
3,Fouls,2238
4,Goalkeeper Saves,2238
5,Offsides,2238
6,Passes %,2238
7,Passes accurate,2238
8,Red Cards,2238
9,Shots insidebox,2238


## 12. Small analytical smoke tests

In [18]:
con.sql(
    """
    SELECT
        f.fixture_id,
        f.match_date,
        ht.team_name AS home_team,
        aw.team_name AS away_team,
        f.home_goals,
        f.away_goals,
        c.competition_name,
        f.season
    FROM fixtures f
    JOIN teams ht
        ON f.home_team_id = ht.team_id
    JOIN teams aw
        ON f.away_team_id = aw.team_id
    JOIN competitions c
        ON f.competition_id = c.competition_id
    ORDER BY f.match_date
    LIMIT 10
    """
).df()

,fixture_id,match_date,home_team,away_team,home_goals,away_goals,competition_name,season
0,891275,2022-07-06 10:00:00,Oliveirense,SC Braga,2,6,Friendlies Clubs,2022
1,922192,2022-07-09 00:00:00,FC Porto,Bristol Rovers,3,0,Friendlies Clubs,2022
2,891344,2022-07-09 09:00:00,Vizela,SC Braga,2,3,Friendlies Clubs,2022
3,870633,2022-07-09 15:00:00,Reading,Benfica,0,2,Friendlies Clubs,2022
4,891404,2022-07-12 12:00:00,Arouca,SC Braga,<NA>,<NA>,Friendlies Clubs,2022
5,890533,2022-07-12 17:00:00,SC Braga,Arouca,3,2,Friendlies Clubs,2022
6,922224,2022-07-13 18:00:00,Sporting CP,Union St. Gilloise,1,1,Friendlies Clubs,2022
7,890578,2022-07-13 18:00:00,Sporting CP,Union St. Gilloise,1,1,Friendlies Clubs,2022
8,890588,2022-07-14 09:30:00,Portimonense,FC Porto,0,1,Friendlies Clubs,2022
9,922235,2022-07-14 09:30:00,Portimonense,FC Porto,0,1,Friendlies Clubs,2022


In [19]:
con.sql(
    """
    SELECT
        ft.fixture_id,
        t.team_name,
        o.team_name AS opponent,
        ft.side,
        ft.goals_for,
        ft.goals_against,
        ft.result,
        ft.points
    FROM fixture_teams ft
    JOIN teams t ON ft.team_id = t.team_id
    JOIN teams o ON ft.opponent_id = o.team_id
    ORDER BY ft.fixture_id, ft.side
    LIMIT 10
    """
).df()

,fixture_id,team_name,opponent,side,goals_for,goals_against,result,points
0,862183,Tondela,FC Porto,away,0,3,L,0
1,862183,FC Porto,Tondela,home,3,0,W,3
2,870633,Benfica,Reading,away,2,0,W,3
3,870633,Reading,Benfica,home,0,2,L,0
4,870734,Middlesbrough,SC Braga,away,<NA>,<NA>,NaN,<NA>
5,870734,SC Braga,Middlesbrough,home,<NA>,<NA>,NaN,<NA>
6,871003,Newcastle,Benfica,away,2,3,L,0
7,871003,Benfica,Newcastle,home,3,2,W,3
8,889896,AS Roma,Sporting CP,away,2,3,L,0
9,889896,Sporting CP,AS Roma,home,3,2,W,3


In [20]:
con.sql(
    """
    SELECT *
    FROM team_match_statistics_wide
    ORDER BY fixture_id, team_id
    LIMIT 10
    """
).df()

,fixture_id,team_id,shots_on_goal,shots_off_goal,total_shots,blocked_shots,shots_inside_box,shots_outside_box,fouls,corner_kicks,offsides,ball_possession,yellow_cards,red_cards,goalkeeper_saves,total_passes,passes_accurate,passes_pct,expected_goals,goals_prevented
0,898604,226,0.0,4.0,7.0,3.0,1.0,6.0,19.0,6.0,1.0,65.0,7.0,NaN,6.0,534.0,452.0,85.0,NaN,NaN
1,898604,810,7.0,4.0,12.0,1.0,10.0,2.0,16.0,7.0,2.0,35.0,4.0,NaN,0.0,286.0,209.0,73.0,NaN,NaN
2,898605,211,6.0,12.0,22.0,4.0,13.0,9.0,7.0,7.0,2.0,73.0,3.0,0.0,2.0,759.0,662.0,87.0,NaN,NaN
3,898605,240,2.0,0.0,3.0,1.0,1.0,2.0,7.0,1.0,2.0,27.0,1.0,1.0,2.0,276.0,192.0,70.0,NaN,NaN
4,898606,216,8.0,7.0,18.0,3.0,12.0,6.0,13.0,7.0,0.0,63.0,3.0,NaN,4.0,573.0,468.0,82.0,NaN,NaN
5,898606,222,5.0,1.0,10.0,4.0,6.0,4.0,11.0,1.0,1.0,37.0,1.0,NaN,8.0,328.0,233.0,71.0,NaN,NaN
6,898607,234,5.0,4.0,12.0,3.0,7.0,5.0,19.0,6.0,3.0,52.0,6.0,NaN,4.0,416.0,344.0,83.0,NaN,NaN
7,898607,762,5.0,5.0,17.0,7.0,6.0,11.0,17.0,8.0,5.0,48.0,1.0,NaN,5.0,380.0,305.0,80.0,NaN,NaN
8,898608,230,5.0,4.0,10.0,1.0,4.0,6.0,14.0,1.0,0.0,47.0,4.0,NaN,5.0,367.0,278.0,76.0,NaN,NaN
9,898608,242,5.0,8.0,15.0,2.0,7.0,8.0,11.0,2.0,2.0,53.0,1.0,NaN,3.0,405.0,333.0,82.0,NaN,NaN


## 13. Final summary and next stage

After this notebook succeeds, the project has the following reproducible flow:

`raw API JSON -> validated transformation -> DuckDB relational core -> analytical views`

### Source-quality rules now enforced

- API entity IDs must be positive integers to be treated as real identities;
- `0` is treated as a missing identifier rather than a valid entity ID;
- missing lineup and player-statistic IDs are recovered only when
  `team_id + normalized player_name` maps to exactly one known real player;
- ambiguous or unresolved rows are excluded from the relational core and kept
  in audit DataFrames;
- no synthetic player or coach IDs are generated;
- a valid `coach_id` may coexist with a null `coach_name`.

Physical tables:

- `competitions`
- `competition_seasons`
- `teams`
- `players`
- `coaches`
- `venues`
- `fixtures`
- `events`
- `team_match_statistics`
- `lineups`
- `lineup_players`
- `player_match_statistics`

Analytical views:

- `fixture_teams`
- `team_match_statistics_wide`

Transformation-quality artifacts:

- `missing_lineup_player_ids_df`
- `lineup_players_processing_df`
- `lineup_player_issues_df`
- `player_match_statistics_processing_df`
- `player_match_statistics_issues_df`
- `transformation_quality_summary_df`

The next notebook should work from DuckDB rather than return to the raw JSON
layer. A sensible next stage is data-quality and analytical-coverage analysis
by competition, season, team, fixture status, and metric availability before
beginning the Braga-focused football analysis.


In [21]:
final_summary = con.sql(
    """
    SELECT 'fixtures' AS object, COUNT(*) AS rows FROM fixtures
    UNION ALL
    SELECT 'fixture_teams', COUNT(*) FROM fixture_teams
    UNION ALL
    SELECT 'events', COUNT(*) FROM events
    UNION ALL
    SELECT 'team_match_statistics', COUNT(*) FROM team_match_statistics
    UNION ALL
    SELECT 'player_match_statistics', COUNT(*) FROM player_match_statistics
    """
).df()

final_summary

,object,rows
0,fixtures,1222
1,fixture_teams,2444
2,events,21214
3,team_match_statistics,38432
4,player_match_statistics,44191
